# 02-Bias-Analysis

After auditing the data for quality issues the next steps is to detect any bias patterns when assigning credit scores based on the EU AI Act. Credit scoring systems are classified as high-risk AI under EU AI Act Annex III. The following obligations have to be considered:

* Art. 10 (data governance)
* Art. 29 (human oversight)
* Art. 71 (penalties up to €30M or 6% global turnover for violations)

The following anaylsis implements mandatory bias monitoring, disparate impact analysis,
and fairness metrics required for high-risk AI systems operating in the EU.

In addition the EEOC Four-Fifths (80%) rule will be applied to identified potential bias categories. Under the AI Act (Art. 10 §5) and classical US fair-lending doctrine, a group whose approval rate falls below 80% of the highest group's rate is considered to suffer a disparate impact, triggering mandatory investigation.


## Setup and Connection

In [1]:
import json
from pathlib import Path
from pymongo import MongoClient

In [2]:
# Connect to local MongoDB
client = MongoClient('localhost', 27017)

# Create/access database and collection
db = client['project']
collection = db['credit_applications']
print("Connected to MongoDB successfully!")

Connected to MongoDB successfully!


In [3]:
# Load the json file
current_dir = Path.cwd()
repo_root = current_dir.parent
json_path = repo_root / "data" / "clean_credit_applications.json"

with open(json_path, 'r') as file:
    data = json.load(file)

print(f"Successfully loaded {len(data)} records from {json_path.name}")

Successfully loaded 499 records from clean_credit_applications.json


## Bias Analysis

### 01-Gender Bias

In [5]:
# Run the aggregation to get approval rates by gender
gender_pipeline = [
    {
        "$addFields": {
            "gender_norm": {
                "$switch": {
                    "branches": [
                        {
                            "case": {"$in": [
                                "$applicant_info.gender",
                                ["Male", "M"]
                            ]},
                            "then": "Male"
                        },
                        {
                            "case": {"$in": [
                                "$applicant_info.gender",
                                ["Female", "F"]
                            ]},
                            "then": "Female"
                        },
                    ],
                    "default": "Unknown"
                }
            }
        }
    },
    {
        "$group": {
            "_id": "$gender_norm",
            "total": {"$sum": 1},
            "approved": {
                "$sum": {
                    "$cond": ["$decision.loan_approved", 1, 0]
                }
            },
            "avg_interest_rate": {
                "$avg": {
                    "$cond": [
                        "$decision.loan_approved",
                        "$decision.interest_rate",
                        None
                    ]
                }
            },
            "avg_approved_amount": {
                "$avg": {
                    "$cond": [
                        "$decision.loan_approved",
                        "$decision.approved_amount",
                        None
                    ]
                }
            }
        }
    },
    {
        "$addFields": {
            "approval_rate": {
                "$divide": ["$approved", "$total"]
            }
        }
    },
    {"$sort": {"approval_rate": -1}}
]

gender_results = list(collection.aggregate(gender_pipeline))

print("\n  Approval Rates & Pricing by Gender:")
print(f"  {'Gender':<12} {'Total':>6} {'Approved':>9} "
        f"{'Rate':>8} {'Avg APR':>9} {'Avg Amount':>12}")
print("  " + "-" * 58)

for g in gender_results:
    apr = (f"{g['avg_interest_rate']:.2f}%"
               if g["avg_interest_rate"] else "N/A")
    amt = (f"${g['avg_approved_amount']:,.0f}"
               if g["avg_approved_amount"] else "N/A")
    print(f"  {str(g['_id']):<12} {g['total']:>6} {g['approved']:>9} "
              f"{g['approval_rate'] * 100:>7.1f}% {apr:>9} {amt:>12}")


  Approval Rates & Pricing by Gender:
  Gender        Total  Approved     Rate   Avg APR   Avg Amount
  ----------------------------------------------------------
  Male            244       162    66.4%     4.62%      $48,938
  Female          248       127    51.2%     4.49%      $46,669


In [53]:
# Apply the EEOC Four-Fifths (80%) rule

# Under the AI Act (Art. 10 §5) and classical US fair-lending doctrine,
# a group whose approval rate falls below 80% of the highest group's
# rate is considered to suffer a disparate impact, triggering mandatory
# investigation.

print("\n  Disparate Impact Analysis (80% Rule):")

if len(gender_results) < 2:
    print("Insufficient groups for disparate-impact analysis.")

else:
    # Build a dictionary of group label -> approval rate
    rates = {r["_id"]: r["approval_rate"] for r in gender_results}

    # Identify the reference group (highest approval rate)
    max_rate = max(rates.values())
    max_group = max(rates, key=rates.get)

    print(f"\n  Reference group (highest rate): {max_group} "
          f"→ {max_rate * 100:.1f}%")
    print(f"  80% threshold: {max_rate * 0.8 * 100:.1f}%\n")

    # Compare each group against the 80% threshold
    summary = {}

    for group, rate in sorted(rates.items(), key=lambda x: -x[1]):
        ratio = rate / max_rate if max_rate > 0 else 0
        passed = ratio >= 0.8
        status = "PASS" if passed else "FAIL — INVESTIGATE"

        print(f"  {str(group):<20} rate: {rate * 100:5.1f}%  "
              f"ratio: {ratio:.3f}  {status}")

        summary[group] = {"rate": rate, "ratio": ratio, "pass": passed}

gender_summary = summary


  Disparate Impact Analysis (80% Rule):

  Reference group (highest rate): Male → 66.4%
  80% threshold: 53.1%

  Male                 rate:  66.4%  ratio: 1.000  PASS
  Female               rate:  51.2%  ratio: 0.771  FAIL — INVESTIGATE


### 02-Age Bias

In [6]:
# Age is a protected characteristic under:
#   - Age Discrimination Act / ECOA
#   - EU AI Act Art. 10 (prohibited use of age as discriminatory proxy)
# Note: Minors (<18) are flagged as a data quality issue since they cannot
# hold credit contracts under the AI Act.

# Fetch only the fields needed for age bias analysis
raw_docs = list(collection.find(
    {},
    {
        "applicant_info.date_of_birth": 1,
        "applicant_info.age": 1,
        "decision.loan_approved": 1,
        "decision.interest_rate": 1
    }
))

# Build bucket counts in Python to handle multi-format DOB
bucket_order = ["<18", "18-24", "25-34", "35-44",
                "45-54", "55-64", "65+", "Unknown"]

buckets = {
    b: {"total": 0, "approved": 0, "interest_rates": []}
    for b in bucket_order
}

minor_ids = []

for doc in raw_docs:
    age = (doc.get("applicant_info") or {}).get("age")

    # Assign age band
    if age is None:
        age_band = "Unknown"
    elif age < 18:
        age_band = "<18"
    elif age < 25:
        age_band = "18-24"
    elif age < 35:
        age_band = "25-34"
    elif age < 45:
        age_band = "35-44"
    elif age < 55:
        age_band = "45-54"
    elif age < 65:
        age_band = "55-64"
    else:
        age_band = "65+"

    # Accumulate bucket stats
    buckets[age_band]["total"] += 1

    approved = (doc.get("decision") or {}).get("loan_approved", False)
    if approved:
        buckets[age_band]["approved"] += 1

    interest_rate = (doc.get("decision") or {}).get("interest_rate")
    if interest_rate is not None:
        buckets[age_band]["interest_rates"].append(interest_rate)

    # Flag minors
    if age is not None and age < 18:
        minor_ids.append(str(doc.get("_id")))

# Report minors
if minor_ids:
    print(f"\nDATA QUALITY — Apparent minors (<18) detected: "
          f"{len(minor_ids)} record(s). IDs: {minor_ids}")
    print("  Action: Verify DOB; minors cannot hold credit contracts "
          "(AI Act Art. 10 data quality obligation).")

# Print approval rate table
print(f"\n  {'Age Band':<12} {'Total':>6} {'Approved':>9} "
      f"{'Rate':>8} {'Avg APR':>9}")
print("  " + "-" * 47)

results = []

for band in bucket_order:
    b = buckets[band]

    if b["total"] == 0:
        continue

    rate = b["approved"] / b["total"]
    avg_apr = (
        sum(b["interest_rates"]) / len(b["interest_rates"])
        if b["interest_rates"] else None
    )
    apr_str = f"{avg_apr:.2f}%" if avg_apr else "N/A"

    print(f"  {band:<12} {b['total']:>6} {b['approved']:>9} "
          f"{rate * 100:>7.1f}% {apr_str:>9}")

    results.append({
        "_id": band,
        "total": b["total"],
        "approved": b["approved"],
        "approval_rate": rate
    })


  Age Band      Total  Approved     Rate   Avg APR
  -----------------------------------------------
  18-24            11         5    45.5%     4.56%
  25-34           140        63    45.0%     4.39%
  35-44           167       109    65.3%     4.51%
  45-54            83        54    65.1%     4.66%
  55-64            52        32    61.5%     4.51%
  65+              12         7    58.3%     5.37%
  Unknown          27        19    70.4%     4.99%


In [7]:
# Apply the EEOC Four-Fifths (80%) Rule
print("\n  Disparate Impact Analysis (80% Rule):")

if len(results) < 2:
    print("Insufficient groups for disparate-impact analysis.")

else:
    rates = {r["_id"]: r["approval_rate"] for r in results}
    max_rate = max(rates.values())
    max_group = max(rates, key=rates.get)

    print(f"\n  Reference group (highest rate): {max_group} "
          f"→ {max_rate * 100:.1f}%")
    print(f"  80% threshold: {max_rate * 0.8 * 100:.1f}%\n")

    summary = {}

    for r in sorted(results, key=lambda x: -x["approval_rate"]):
        group = r["_id"]
        rate = r["approval_rate"]
        ratio = rate / max_rate if max_rate > 0 else 0
        passed = ratio >= 0.8
        status = "PASS" if passed else "FAIL — INVESTIGATE"

        print(f"  {str(group):<20} rate: {rate * 100:5.1f}%  "
              f"ratio: {ratio:.3f}  {status}")

        summary[group] = {
            "rate": rate,
            "ratio": ratio,
            "pass": passed
        }
age_summary = summary


  Disparate Impact Analysis (80% Rule):

  Reference group (highest rate): Unknown → 70.4%
  80% threshold: 56.3%

  Unknown              rate:  70.4%  ratio: 1.000  PASS
  35-44                rate:  65.3%  ratio: 0.928  PASS
  45-54                rate:  65.1%  ratio: 0.925  PASS
  55-64                rate:  61.5%  ratio: 0.874  PASS
  65+                  rate:  58.3%  ratio: 0.829  PASS
  18-24                rate:  45.5%  ratio: 0.646  FAIL — INVESTIGATE
  25-34                rate:  45.0%  ratio: 0.639  FAIL — INVESTIGATE


### 03-Geographic Bias

In [8]:
# Zip-code-based lending bias is a modern form of redlining. The AI Act
# (Recital 44, Art. 10) prohibits AI systems from using proxies that
# reproduce or amplify historical discrimination. Geographic clustering
# in the NovaCred dataset (LA 90xxx, NYC 10xxx, Atlanta 30xxx) may
# proxy for race/ethnicity and must be audited.

# Aggregate approval stats by zip code
pipeline_geo = [
    {
        "$group": {
            "_id": "$applicant_info.zip_code",
            "total": {"$sum": 1},
            "approved": {
                "$sum": {
                    "$cond": ["$decision.loan_approved", 1, 0]
                }
            },
            "avg_interest_rate": {
                "$avg": {
                    "$cond": [
                        "$decision.loan_approved",
                        "$decision.interest_rate",
                        None
                    ]
                }
            }
        }
    }
]

zip_results = list(collection.aggregate(pipeline_geo))

# Roll up zip codes into regions
region_buckets = {}

for z in zip_results:
    zip_code = z["_id"]
    zip_str = str(zip_code).strip() if zip_code is not None else None

    # Infer region from zip prefix
    if zip_str is None:
        region = "Unknown"
    elif zip_str.startswith(("90", "91")):
        region = "Los Angeles, CA"
    elif zip_str.startswith("10"):
        region = "New York City, NY"
    elif zip_str.startswith("30"):
        region = "Atlanta, GA"
    else:
        region = "Other"

    if region not in region_buckets:
        region_buckets[region] = {
            "total": 0,
            "approved": 0,
            "interest_rates": [],
            "_id": region
        }

    region_buckets[region]["total"] += z["total"]
    region_buckets[region]["approved"] += z["approved"]

    if z["avg_interest_rate"]:
        region_buckets[region]["interest_rates"].append(
            z["avg_interest_rate"]
        )

# Print approval rate table
print(f"\n  {'Region':<22} {'Total':>6} {'Approved':>9} "
      f"{'Rate':>8} {'Avg APR':>9}")
print("  " + "-" * 57)

results = []

for region, b in sorted(
    region_buckets.items(),
    key=lambda x: -x[1]["approved"] / x[1]["total"]
):
    rate = b["approved"] / b["total"] if b["total"] > 0 else 0
    avg_apr = (
        sum(b["interest_rates"]) / len(b["interest_rates"])
        if b["interest_rates"] else None
    )
    apr_str = f"{avg_apr:.2f}%" if avg_apr else "N/A"

    print(f"  {region:<22} {b['total']:>6} {b['approved']:>9} "
          f"{rate * 100:>7.1f}% {apr_str:>9}")

    results.append({
        "_id": region,
        "total": b["total"],
        "approved": b["approved"],
        "approval_rate": rate
    })


  Region                  Total  Approved     Rate   Avg APR
  ---------------------------------------------------------
  New York City, NY         247       160    64.8%     4.44%
  Atlanta, GA                18        10    55.6%     4.89%
  Los Angeles, CA           227       119    52.4%     4.38%


In [9]:
# Apply the EEOC Four-Fifths (80%) Rule
print("\n  Disparate Impact Analysis (80% Rule):")

if len(results) < 2:
    print("Insufficient groups for disparate-impact analysis.")

else:
    rates = {r["_id"]: r["approval_rate"] for r in results}
    max_rate = max(rates.values())
    max_group = max(rates, key=rates.get)

    print(f"\n  Reference group (highest rate): {max_group} "
          f"→ {max_rate * 100:.1f}%")
    print(f"  80% threshold: {max_rate * 0.8 * 100:.1f}%\n")

    summary = {}

    for r in sorted(results, key=lambda x: -x["approval_rate"]):
        group = r["_id"]
        rate = r["approval_rate"]
        ratio = rate / max_rate if max_rate > 0 else 0
        passed = ratio >= 0.8
        status = "PASS" if passed else "FAIL — INVESTIGATE"

        print(f"  {str(group):<20} rate: {rate * 100:5.1f}%  "
              f"ratio: {ratio:.3f}  {status}")

        summary[group] = {
            "rate": rate,
            "ratio": ratio,
            "pass": passed
        }
        
geo_summary = summary


  Disparate Impact Analysis (80% Rule):

  Reference group (highest rate): New York City, NY → 64.8%
  80% threshold: 51.8%

  New York City, NY    rate:  64.8%  ratio: 1.000  PASS
  Atlanta, GA          rate:  55.6%  ratio: 0.858  PASS
  Los Angeles, CA      rate:  52.4%  ratio: 0.809  PASS


### 04-Spending Category Bias

In [10]:
# The dataset includes categories such as 'Gambling', 'Adult Entertainment',
# and 'Alcohol'. Using such categories as model inputs may constitute
# discrimination by proxy (AI Act Art. 10 §5, Recital 44)

sensitive_categories = ["Gambling", "Adult Entertainment", "Alcohol"]

pipeline_spending = [
    {"$unwind": "$spending_behavior"},
    {
        "$group": {
            "_id": "$spending_behavior.category",
            "total_applicants": {"$sum": 1},
            "approved": {
                "$sum": {
                    "$cond": ["$decision.loan_approved", 1, 0]
                }
            },
            "avg_amount": {
                "$avg": "$spending_behavior.amount"
            }
        }
    },
    {
        "$addFields": {
            "approval_rate": {
                "$divide": ["$approved", "$total_applicants"]
            }
        }
    },
    {"$sort": {"approval_rate": 1}}
]

spending_results = list(collection.aggregate(pipeline_spending))

# Print spending category table
print(f"\n  {'Category':<25} {'Applications':>12} "
      f"{'Approved':>9} {'Rate':>8} {'Avg Spend':>11}")
print("  " + "-" * 67)

flagged = []

for cat in spending_results:
    flag = " SENSITIVE" if cat["_id"] in sensitive_categories else ""

    if cat["_id"] in sensitive_categories:
        flagged.append(cat)

    print(f"  {str(cat['_id']):<25} {cat['total_applicants']:>12} "
          f"{cat['approved']:>9} {cat['approval_rate'] * 100:>7.1f}% "
          f"${cat['avg_amount']:>9.0f}{flag}")

# Flag sensitive categories
if flagged:
    print("\n Sensitive spending categories detected "
          "as potential model inputs:")

    for cat in flagged:
        print(f"     • '{cat['_id']}': {cat['approval_rate'] * 100:.1f}% "
              f"approval rate across {cat['total_applicants']} applicants.")

    print("  Action: Verify that 'Gambling', 'Adult Entertainment', and "
          "'Alcohol' spending\n"
          "  are NOT used as model features. If so, remove per Art. 10 §5 "
          "(prohibited\n"
          "  data use) and re-train. Document in Technical Documentation "
          "(Art. 11).")


  Category                  Applications  Approved     Rate   Avg Spend
  -------------------------------------------------------------------
  Rent                                59        26    44.1% $      524
  Dining                              63        30    47.6% $      472
  Gambling                             6         3    50.0% $      408 SENSITIVE
  Fitness                             70        36    51.4% $      454
  Entertainment                       70        40    57.1% $      491
  Healthcare                          68        39    57.4% $      451
  Education                           64        37    57.8% $      500
  Travel                              80        48    60.0% $      493
  Adult Entertainment                  5         3    60.0% $      591 SENSITIVE
  Groceries                           63        38    60.3% $      487
  Transportation                      59        37    62.7% $      447
  Utilities                           74        48    64

### 05-Rejection Fairness

In [11]:
# The rejection reason 'algorithm_risk_score' is opaque and may mask bias.
# Under AI Act Art. 13 (Transparency) and Art. 14 (Human Oversight),
# applicants must receive meaningful explanations for adverse decisions.

pipeline_rejection = [
    {"$match": {"decision.loan_approved": False}},
    {
        "$addFields": {
            "gender_norm": {
                "$switch": {
                    "branches": [
                        {
                            "case": {"$in": [
                                "$applicant_info.gender",
                                ["Male", "M"]
                            ]},
                            "then": "Male"
                        },
                        {
                            "case": {"$in": [
                                "$applicant_info.gender",
                                ["Female", "F"]
                            ]},
                            "then": "Female"
                        },
                    ],
                    "default": "Unknown"
                }
            }
        }
    },
    {
        "$group": {
            "_id": {
                "gender": "$gender_norm",
                "reason": "$decision.rejection_reason"
            },
            "count": {"$sum": 1}
        }
    },
    {"$sort": {"_id.gender": 1, "count": -1}}
]

rejection_results = list(collection.aggregate(pipeline_rejection))

# Restructure results for display
gender_rejections = {}

for r in rejection_results:
    gender = r["_id"]["gender"]
    reason = r["_id"]["reason"] or "Unknown"

    if gender not in gender_rejections:
        gender_rejections[gender] = {}

    gender_rejections[gender][reason] = r["count"]

# Print rejection reason table
print("\n  Rejection reasons by gender:\n")

all_reasons = sorted({
    reason
    for g in gender_rejections.values()
    for reason in g
})

header = f"  {'Reason':<35}"
for g in sorted(gender_rejections.keys()):
    header += f" {g:>10}"
print(header)
print("  " + "-" * (35 + 11 * len(gender_rejections)))

for reason in all_reasons:
    row = f"  {reason:<35}"
    for g in sorted(gender_rejections.keys()):
        count = gender_rejections.get(g, {}).get(reason, 0)
        row += f" {count:>10}"
    print(row)

# Flag opaque algorithmic rejections
print("\n AI ACT ART. 13 CHECK — 'algorithm_risk_score' rejections:")

for gender, reasons in gender_rejections.items():
    total = sum(reasons.values())
    algo = reasons.get("algorithm_risk_score", 0)
    algo_pct = algo / total * 100 if total > 0 else 0
    flag = " HIGH" if algo_pct > 50 else ""

    print(f"    {gender}: {algo}/{total} = {algo_pct:.1f}% "
          f"opaque algorithmic rejections{flag}")

print("\n  Action: If 'algorithm_risk_score' represents >50% of rejections")
print("  for any protected group, conduct root-cause analysis and")
print("  implement human review (Art. 14 Human Oversight obligation).")


  Rejection reasons by gender:

  Reason                                  Female       Male
  ---------------------------------------------------------
  algorithm_risk_score                        97         67
  high_dti_ratio                               8          4
  insufficient_credit_history                 15          8
  low_income                                   1          3

 AI ACT ART. 13 CHECK — 'algorithm_risk_score' rejections:
    Female: 97/121 = 80.2% opaque algorithmic rejections HIGH
    Male: 67/82 = 81.7% opaque algorithmic rejections HIGH

  Action: If 'algorithm_risk_score' represents >50% of rejections
  for any protected group, conduct root-cause analysis and
  implement human review (Art. 14 Human Oversight obligation).


### 06-Intersectional Bias: Gender and Age

In [12]:
# The AI Act (Recital 44) recognises that discrimination often operates
# at the intersection of protected characteristics. A system may appear
# fair on each dimension individually while still discriminating against
# specific subgroups (e.g., young women or older men).
# This section applies the 80% rule to gender × age intersections.


# Fetch only the fields needed for intersectional analysis
raw_docs = list(collection.find(
    {},
    {
        "applicant_info.gender": 1,
        "applicant_info.age": 1,
        "decision.loan_approved": 1
    }
))

# Build intersectional buckets (gender × age band)
intersect_buckets = {}

for doc in raw_docs:
    info = doc.get("applicant_info") or {}

    # Normalise gender inline
    raw_gender = info.get("gender", "")
    if raw_gender in ("Male", "M"):
        gender = "Male"
    elif raw_gender in ("Female", "F"):
        gender = "Female"
    else:
        gender = "Unknown"

    # Assign age band inline
    age = info.get("age")
    if age is None:
        age_band = "Unknown"
    elif age < 18:
        age_band = "<18"
    elif age < 25:
        age_band = "18-24"
    elif age < 35:
        age_band = "25-34"
    elif age < 45:
        age_band = "35-44"
    elif age < 55:
        age_band = "45-54"
    elif age < 65:
        age_band = "55-64"
    else:
        age_band = "65+"

    key = f"{gender} / {age_band}"

    if key not in intersect_buckets:
        intersect_buckets[key] = {"total": 0, "approved": 0}

    intersect_buckets[key]["total"] += 1

    if (doc.get("decision") or {}).get("loan_approved", False):
        intersect_buckets[key]["approved"] += 1

# Print intersectional approval rate table
print(f"\n  {'Subgroup':<25} {'Total':>6} {'Approved':>9} {'Rate':>8}")
print("  " + "-" * 50)

results = []

for key, b in sorted(
    intersect_buckets.items(),
    key=lambda x: -x[1]["approved"] / x[1]["total"]
    if x[1]["total"] > 0 else 0
):
    # Skip very small cells — statistically unreliable
    if b["total"] < 5:
        continue

    rate = b["approved"] / b["total"]

    print(f"  {key:<25} {b['total']:>6} {b['approved']:>9} "
          f"{rate * 100:>7.1f}%")

    results.append({
        "_id": key,
        "total": b["total"],
        "approved": b["approved"],
        "approval_rate": rate
    })


  Subgroup                   Total  Approved     Rate
  --------------------------------------------------
  Male / Unknown                15        11    73.3%
  Male / 35-44                  87        62    71.3%
  Male / 55-64                  24        17    70.8%
  Male / 45-54                  44        30    68.2%
  Female / Unknown              12         8    66.7%
  Female / 45-54                39        24    61.5%
  Female / 35-44                80        47    58.8%
  Male / 25-34                  65        37    56.9%
  Female / 55-64                28        15    53.6%
  Female / 18-24                 6         3    50.0%
  Female / 65+                   8         4    50.0%
  Male / 18-24                   5         2    40.0%
  Female / 25-34                75        26    34.7%


In [13]:
# Apply the EEOC Four-Fifths (80%) Rule
print("\n  Disparate Impact Analysis (80% Rule):")

if len(results) < 2:
    print("Insufficient groups for disparate-impact analysis.")

else:
    rates = {r["_id"]: r["approval_rate"] for r in results}
    max_rate = max(rates.values())
    max_group = max(rates, key=rates.get)

    print(f"\n  Reference group (highest rate): {max_group} "
          f"→ {max_rate * 100:.1f}%")
    print(f"  80% threshold: {max_rate * 0.8 * 100:.1f}%\n")

    summary = {}

    for r in sorted(results, key=lambda x: -x["approval_rate"]):
        group = r["_id"]
        rate = r["approval_rate"]
        ratio = rate / max_rate if max_rate > 0 else 0
        passed = ratio >= 0.8
        status = "PASS" if passed else "FAIL — INVESTIGATE"

        print(f"  {str(group):<25} rate: {rate * 100:5.1f}%  "
              f"ratio: {ratio:.3f}  {status}")

        summary[group] = {
            "rate": rate,
            "ratio": ratio,
            "pass": passed
        }
intersect_summary = summary


  Disparate Impact Analysis (80% Rule):

  Reference group (highest rate): Male / Unknown → 73.3%
  80% threshold: 58.7%

  Male / Unknown            rate:  73.3%  ratio: 1.000  PASS
  Male / 35-44              rate:  71.3%  ratio: 0.972  PASS
  Male / 55-64              rate:  70.8%  ratio: 0.966  PASS
  Male / 45-54              rate:  68.2%  ratio: 0.930  PASS
  Female / Unknown          rate:  66.7%  ratio: 0.909  PASS
  Female / 45-54            rate:  61.5%  ratio: 0.839  PASS
  Female / 35-44            rate:  58.8%  ratio: 0.801  PASS
  Male / 25-34              rate:  56.9%  ratio: 0.776  FAIL — INVESTIGATE
  Female / 55-64            rate:  53.6%  ratio: 0.731  FAIL — INVESTIGATE
  Female / 18-24            rate:  50.0%  ratio: 0.682  FAIL — INVESTIGATE
  Female / 65+              rate:  50.0%  ratio: 0.682  FAIL — INVESTIGATE
  Male / 18-24              rate:  40.0%  ratio: 0.545  FAIL — INVESTIGATE
  Female / 25-34            rate:  34.7%  ratio: 0.473  FAIL — INVESTIGATE


### Compliance Summary

In [14]:
# EU AI Act Compliance Summary
all_analyses = [
    ("Gender (Art. 10)", gender_summary),
    ("Age (Art. 10)", age_summary),
    ("Geography / Redlining", geo_summary),
    ("Intersectional (Gender × Age)", intersect_summary),
]

any_fail = False

# Print compliance table
print(f"\n {'Analysis':<35} {'Status':<20} {'Failing Groups'}")
print("  " + "-" * 75)

for label, di_results in all_analyses:
    if not di_results:
        print(f"  {label:<35} {'NO DATA':<20}")
        continue

    failing = [g for g, v in di_results.items() if not v["pass"]]

    if failing:
        any_fail = True
        status = "FAIL"
        failing_str = ", ".join(str(f) for f in failing[:3])
    else:
        status = "PASS"
        failing_str = "—"

    print(f"  {label:<35} {status:<20} {failing_str}")

print()

# Overall verdict
if any_fail:
    print("OVERALL VERDICT: DISPARATE IMPACT DETECTED")
    print()
    print("Required Actions under EU AI Act:")
    print("┌─────────────────────────────────────────────────────────┐")
    print("│ Art. 9  — Update risk management to cover bias risks    │")
    print("│ Art. 10 — Audit training data for discriminatory labels │")
    print("│ Art. 11 — Document findings in Technical Documentation  │")
    print("│ Art. 13 — Provide meaningful rejection explanations     │")
    print("│ Art. 14 — Implement human review for flagged decisions  │")
    print("│ Art. 29 — Notify deployer (NovaCred) of audit results   │")
    print("└─────────────────────────────────────────────────────────┘")
else:
    print("OVERALL VERDICT: No statistically significant "
          "disparate impact detected.")
    print("Recommendation: Continue quarterly monitoring as required "
          "by Art. 9 (ongoing risk management).")

# Audit metadata
print(f"Records audited : {collection.count_documents({})}")

NameError: name 'gender_summary' is not defined